# Aula 4: Features

Criação do notebook "Features".


## Quantidade de Produtos Distintos

Vamos encontar quais são os produtos mais transacionados e criar suas categorias.
Usaremos toda a base até a janela de corte '2026-06-01'. 

In [0]:
%sql
WITH
tb_transacoes AS (
    SELECT 
        t3.DescNomeProduto,
        COUNT(t1.IdTransacao) AS QtdeTransacoes
    FROM workspace.tmw_loyalty.transacoes AS t1

    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao
    
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
    
    WHERE t1.DtCriacao < '2026-06-01' --AND t1.DtCriacao >= '2026-06-01' - interval 28 days
    GROUP BY ALL
) 

SELECT 
    DescNomeProduto,
    QtdeTransacoes,
    ROUND(QtdeTransacoes / SUM(QtdeTransacoes) OVER () , 4) AS pct
FROM tb_transacoes
ORDER BY pct DESC

Criaria então 6 grupos:
1. 'ChatMessage'
2. 'Lista de presença'
3. 'Presença Streak'
4. 'Resgatar Ponei'
5. 'Troca de Pontos StreamElements'
6. Outros


## Features: PONTOS

Vai dar origem a `fs_pontos.sql`.

Cou parametrizar a query para recever `{date}` quando `'2026-06-01'` 

In [0]:
%sql
WITH tb_transacoes AS (
    SELECT
        *,
        DAYOFWEEK(DATE(DtCriacao)) AS DiaDaSemana, -- 1 = Domingo, 7 = Sábado
        DAYOFYEAR(DATE(DtCriacao)) AS DiaDoAno,
        MONTH(DATE(DtCriacao)) AS Mes,
        WEEKOFYEAR(DATE(DtCriacao)) AS SemanaDoAno,
        FLOOR(DATEDIFF('2026-06-01', DATE(DtCriacao)) / 7) + 1 AS SemanaIdx -- 1 = most recent week, 4 = oldest week
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01' AND DtCriacao >= '2026-06-01' - interval 28 days
),


tb_dia AS (
    SELECT
        IdCliente,
        COUNT(*) AS QtdeTotal,
-- Numerador: contagens por cliente × dia da semana
        COUNT(CASE WHEN DiaDaSemana = 1 THEN 1 END) AS QtdeDom,
        COUNT(CASE WHEN DiaDaSemana = 2 THEN 1 END) AS QtdeSeg,
        COUNT(CASE WHEN DiaDaSemana = 3 THEN 1 END) AS QtdeTer,
        COUNT(CASE WHEN DiaDaSemana = 4 THEN 1 END) AS QtdeQua,
        COUNT(CASE WHEN DiaDaSemana = 5 THEN 1 END) AS QtdeQui,
        COUNT(CASE WHEN DiaDaSemana = 6 THEN 1 END) AS QtdeSex,
        COUNT(CASE WHEN DiaDaSemana = 7 THEN 1 END) AS QtdeSab,

-- Numerador: contagens por cliente × semana relativa
        COUNT(CASE WHEN SemanaIdx = 1 THEN 1 END) AS QtdeSem1,
        COUNT(CASE WHEN SemanaIdx = 2 THEN 1 END) AS QtdeSem2,
        COUNT(CASE WHEN SemanaIdx = 3 THEN 1 END) AS QtdeSem3,
        COUNT(CASE WHEN SemanaIdx = 4 THEN 1 END) AS QtdeSem4
    FROM tb_transacoes
    GROUP BY IdCliente
),

tb_sasonalidade AS (
SELECT
    IdCliente,
    QtdeTotal,

    -- ---------- Shares por dia da semana (%) ----------
    ROUND(QtdeDom * 100.0 / QtdeTotal, 2) AS ShareDomingo,
    ROUND(QtdeSeg * 100.0 / QtdeTotal, 2) AS ShareSegunda,
    ROUND(QtdeTer * 100.0 / QtdeTotal, 2) AS ShareTerca,
    ROUND(QtdeQua * 100.0 / QtdeTotal, 2) AS ShareQuarta,
    ROUND(QtdeQui * 100.0 / QtdeTotal, 2) AS ShareQuinta,
    ROUND(QtdeSex * 100.0 / QtdeTotal, 2) AS ShareSexta,
    ROUND(QtdeSab * 100.0 / QtdeTotal, 2) AS ShareSabado,

    -- ---------- Shares por semana relativa (%) ----------
    ROUND(QtdeSem1 * 100.0 / QtdeTotal, 2) AS ShareSemana1,  -- mais recente
    ROUND(QtdeSem2 * 100.0 / QtdeTotal, 2) AS ShareSemana2,
    ROUND(QtdeSem3 * 100.0 / QtdeTotal, 2) AS ShareSemana3,
    ROUND(QtdeSem4 * 100.0 / QtdeTotal, 2) AS ShareSemana4  -- mais antiga

FROM tb_dia
),

tb_cliente_agg AS (

    SELECT IdCliente,
        COUNT(distinct date(DtCriacao)) AS QtdeFrequencia,
        SUM(QtdePontos) AS QtdePontos,
        SUM(CASE WHEN QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS QtdePontosPositivos,
        MIN(date_diff('2026-06-01', DtCriacao)) AS Recencia,
        COUNT(idTransacao) AS QtdeTransacoes
    FROM tb_transacoes

    GROUP BY ALL
),

tb_cliente_produto AS (
    SELECT 
        t1.IdCliente,
        COUNT(t1.idTransacao) AS QtdeTransacoes,
        -- Percentual de transações por produto
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'ChatMessage' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_chat,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Lista de presença' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_presenca,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_streak,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Resgatar Ponei' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_ponei,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto = 'Troca de Pontos StreamElements' THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_streamElements,
        COUNT(DISTINCT CASE WHEN t3.DescNomeProduto NOT IN ('ChatMessage', 'Lista de presença', 'Presença Streak', 'Resgatar Ponei', 'Troca de Pontos StreamElements') OR t3.DescNomeProduto IS NULL THEN t1.idTransacao ELSE NULL END) / COUNT(DISTINCT t1.idTransacao) AS pctTransacao_outros,
        -- Percentual de pontos por produto
        SUM(CASE WHEN t3.DescNomeProduto = 'ChatMessage' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_chat,
        SUM(CASE WHEN t3.DescNomeProduto = 'Lista de presença' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_presenca,
        SUM(CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_streak,
        SUM(CASE WHEN t3.DescNomeProduto = 'Resgatar Ponei' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_ponei,
        SUM(CASE WHEN t3.DescNomeProduto = 'Troca de Pontos StreamElements' THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) / SUM(ABS(t2.QtdeProduto * t2.vlProduto)) AS pctPontosAbs_streamElements,
        try_divide(SUM(CASE WHEN t3.DescNomeProduto NOT IN ('ChatMessage', 'Lista de presença', 'Presença Streak', 'Resgatar Ponei', 'Troca de Pontos StreamElements') OR t3.DescNomeProduto IS NULL THEN ABS(t2.QtdeProduto * t2.vlProduto) ELSE NULL END) , SUM(ABS(t2.QtdeProduto * t2.vlProduto)) ) AS pctPontosAbs_outros,
        -- Flag de Streak
        MAX( CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN 1 ELSE 0 END) AS flStreak,
        -- Dias desde o último Streak
        MIN(CASE WHEN t3.DescNomeProduto = 'Presença Streak' THEN date_diff('2026-06-01', t1.DtCriacao) ELSE NULL END) AS diasDesdeUltimoStreak,
        -- Quantidade de produto distinto
        COUNT(DISTINCT t2.IdProduto) AS qtdeProdutoDistintos

    FROM tb_transacoes AS t1

    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao
    
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
    
    WHERE t1.DtCriacao < '2026-06-01' AND t1.DtCriacao >= '2026-06-01' - interval 28 days
    GROUP BY ALL
),

tb_vida AS (
    SELECT
        t1.idCliente,
        MAX(date_diff('2026-06-01', t1.dtCriacao)) AS diasPrimeiraTransacao,
        COUNT(DISTINCT DATE(DtCriacao)) AS freqVida,
        SUM(t1.QtdePontos) AS SaldoPontos
    FROM workspace.tmw_loyalty.transacoes AS t1
    WHERE DtCriacao < '2026-06-01'
    GROUP BY ALL
),

tb_join AS (
    SELECT 
        t1.*,
        (t1.QtdePontosPositivos - (SELECT AVG(QtdePontosPositivos) FROM tb_cliente_agg) ) / (SELECT stddev(QtdePontosPositivos) FROM tb_cliente_agg) AS zScore,
        t3.diasPrimeiraTransacao,
        t3.freqVida,
        t3.SaldoPontos,
        t2.pctTransacao_chat,
        t2.pctTransacao_presenca,
        t2.pctTransacao_streak,
        t2.pctTransacao_ponei,
        t2.pctTransacao_streamElements,
        t2.pctTransacao_outros,
        t2.pctPontosAbs_chat,
        t2.pctPontosAbs_presenca,
        t2.pctPontosAbs_streak,
        t2.pctPontosAbs_ponei,
        t2.pctPontosAbs_streamElements,
        t2.pctPontosAbs_outros,
        t2.flStreak,
        t2.diasDesdeUltimoStreak,
        t2.qtdeProdutoDistintos,
        t4.ShareDomingo,
        t4.ShareSegunda,
        t4.ShareTerca,
        t4.ShareQuarta,
        t4.ShareQuinta,
        t4.ShareSexta,
        t4.ShareSabado,
        t4.ShareSemana1,
        t4.ShareSemana2,
        t4.ShareSemana3,
        t4.ShareSemana4

    FROM tb_cliente_agg AS t1

    LEFT JOIN tb_cliente_produto AS t2
        ON t1.idCliente = t2.idCliente

    LEFT JOIN tb_vida AS t3
        ON t1.idCliente = t3.idCliente

    LEFT JOIN tb_sasonalidade AS t4
        ON t1.idCliente = t4.idCliente
)

SELECT
    '2026-06-01' AS dtRef,
    *
FROM tb_join



## Features: CURSOS

Vai dar origem a `fs_cursos.sql`

In [0]:
%sql
WITH 
tb_usario_curso AS (

    SELECT t1.idTMWCliente,
        t2.descSlugCurso,
        min(dtCriacao) AS dtInicioCurso,
        max(dtCriacao) AS dtUltimoEp,
        date_diff(max(dtCriacao), min(dtCriacao)) AS diasEntrePrimeUltimo,
        COUNT(*) AS qtEpCurso

    FROM workspace.tmw_education.usuarios_tmw AS t1

    LEFT JOIN workspace.tmw_education.cursos_episodios_completos AS T2
    on t1.idUsuario = t2.idUsuario

    WHERE t2.dtCriacao < '2026-06-01'

    GROUP BY ALL
),

tb_curso AS (

    SELECT descSlugCurso,
           COUNT(*) AS qtEpsCurso
    FROM workspace.tmw_education.cursos_episodios
    GROUP BY ALL

),

tb_curso_avanco AS (

    SELECT t1.*,
        t1.qtEpCurso / t2.qtepscurso AS pctCursoCompleto

    FROM tb_usario_curso AS t1
    LEFT JOIN tb_curso AS t2
    ON t1.descslugcurso = t2.descslugcurso

),

tb_agg AS (

    SELECT
        idTMWCliente AS idCliente,
        COUNT(distinct descslugcurso) AS qtdCursosIniciados,
        COUNT(distinct CASE WHEN pctCursoCompleto = 1 THEN descslugcurso ELSE NULL END) AS qtdCursosFinalizados,
        SUM(CASE WHEN descslugcurso = 'python-2025' THEN pctCursoCompleto else 0 end) as python_2025,
        SUM(CASE WHEN descslugcurso = 'plataforma-ml-2026' THEN pctCursoCompleto else 0 end) as plataforma_ml_2026,
        SUM(CASE WHEN descslugcurso = 'mlflow-2025' THEN pctCursoCompleto else 0 end) as mlflow_2025,
        SUM(CASE WHEN descslugcurso = 'carreira' THEN pctCursoCompleto else 0 end) as carreira,
        SUM(CASE WHEN descslugcurso = 'nekt-2025' THEN pctCursoCompleto else 0 end) as nekt_2025,
        SUM(CASE WHEN descslugcurso = 'estatistica-2025' THEN pctCursoCompleto else 0 end) as estatistica_2025,
        SUM(CASE WHEN descslugcurso = 'coleta-dados-2024' THEN pctCursoCompleto else 0 end) as coleta_dados_2024,
        SUM(CASE WHEN descslugcurso = 'python-2024' THEN pctCursoCompleto else 0 end) as python_2024,
        SUM(CASE WHEN descslugcurso = 'lago-mago-2024' THEN pctCursoCompleto else 0 end) as lago_mago_2024,
        SUM(CASE WHEN descslugcurso = 'github-2025' THEN pctCursoCompleto else 0 end) as github_2025,
        SUM(CASE WHEN descslugcurso = 'trampar-lakehouse-2024' THEN pctCursoCompleto else 0 end) as trampar_lakehouse_2024,
        SUM(CASE WHEN descslugcurso = 'ds-databricks-2024' THEN pctCursoCompleto else 0 end) as ds_databricks_2024,
        SUM(CASE WHEN descslugcurso = 'sql-2020' THEN pctCursoCompleto else 0 end) as sql_2020,
        SUM(CASE WHEN descslugcurso = 'ds-pontos-2024' THEN pctCursoCompleto else 0 end) as ds_pontos_2024,
        SUM(CASE WHEN descslugcurso = 'streamlit-2025' THEN pctCursoCompleto else 0 end) as streamlit_2025,
        SUM(CASE WHEN descslugcurso = 'estatistica-2024' THEN pctCursoCompleto else 0 end) as estatistica_2024,
        SUM(CASE WHEN descslugcurso = 'ml-2024' THEN pctCursoCompleto else 0 end) as ml_2024,
        SUM(CASE WHEN descslugcurso = 'ragia' THEN pctCursoCompleto else 0 end) as ragia,
        SUM(CASE WHEN descslugcurso = 'sql-2025' THEN pctCursoCompleto else 0 end) as sql_2025,
        SUM(CASE WHEN descslugcurso = 'loyalty-predict-2025' THEN pctCursoCompleto else 0 end) as loyalty_predict_2025,
        SUM(CASE WHEN descslugcurso = 'go-2026' THEN pctCursoCompleto else 0 end) as go_2026,
        SUM(CASE WHEN descslugcurso = 'f1-lake' THEN pctCursoCompleto else 0 end) as f1_lake,
        SUM(CASE WHEN descslugcurso = 'ia-canal-2025' THEN pctCursoCompleto else 0 end) as ia_canal_2025,
        SUM(CASE WHEN descslugcurso = 'tse-analytics-2024' THEN pctCursoCompleto else 0 end) as tse_analytics_2024,
        SUM(CASE WHEN descslugcurso = 'machine-learning-2025' THEN pctCursoCompleto else 0 end) as machine_learning_2025,
        SUM(CASE WHEN descslugcurso = 'matchmaking-trampar-de-casa-2024' THEN pctCursoCompleto else 0 end) as matchmaking_trampar_de_casa_2024,
        SUM(CASE WHEN descslugcurso = 'pandas-2024' THEN pctCursoCompleto else 0 end) as pandas_2024,
        SUM(CASE WHEN descslugcurso = 'pandas-2025' THEN pctCursoCompleto else 0 end) as pandas_2025,
        SUM(CASE WHEN descslugcurso = 'speed-f1' THEN pctCursoCompleto else 0 end) as speed_f1,
        SUM(CASE WHEN descslugcurso = 'github-2024' THEN pctCursoCompleto else 0 end) as github_2024,
        AVG(diasEntrePrimeUltimo) AS avgTempoInicioUltimo,
        AVG(CASE WHEN pctCursoCompleto = 1 THEN diasEntrePrimeUltimo ELSE NULL END) AS avgTempoInicioFim


    FROM tb_curso_avanco

    GROUP BY ALL

)

SELECT '2026-06-01' AS dtRef,
        *

FROM tb_agg

In [0]:
%sql
-- SELECT
--     t2.descSlugCurso,
--     t2.nrAno,
--     MIN(DATE(t1.dtCriacao)) AS Date_min,
--     MAX(DATE(t1.dtCriacao)) AS Date_max
-- FROM workspace.tmw_education.cursos_episodios_completos AS t1
-- FULL JOIN workspace.tmw_education.cursos AS t2
--     ON t1.descslugcurso = t2.descslugcurso
-- GROUP BY ALL

SELECT 
*
FROM workspace.tmw_education.cursos_episodios
WHERE nrEp = 1